In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
"""
Phase 6: Evaluation
Compares retrieval quality between fixed-size and section-aware chunking
strategies across a fixed test query set covering single-document lookups,
cross-year comparisons, and cross-company comparisons.
"""

import mlflow.deployments
import pandas as pd
from databricks.vector_search.client import VectorSearchClient

client = mlflow.deployments.get_deploy_client("databricks")
vsc = VectorSearchClient()

fixed_index = vsc.get_index(endpoint_name="rag_pipeline_endpoint", index_name="rag_pipeline.main.fixed_size_index")
section_index = vsc.get_index(endpoint_name="rag_pipeline_endpoint", index_name="rag_pipeline.main.section_aware_index")

def embed_query(text):
    response = client.predict(
        endpoint="databricks-bge-large-en",
        inputs={"input": [text]}
    )
    return response["data"][0]["embedding"]

def run_query(index, query_vector, num_results=5):
    results = index.similarity_search(
        query_vector=query_vector,
        columns=["chunk_id", "company", "fiscal_year", "chunk_text"],
        num_results=num_results
    )
    return results["result"]["data_array"]


/home/spark-05f7ce02-3ff0-4236-87f9-6d/.ipykernel/73/command-7814296316566072-2518821030:10: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [0]:
# --- Test query set: covers all three target query types ---
test_queries = [
    {"query": "What are NVIDIA's supply chain risk factors?", "type": "single_doc"},
    {"query": "What was Apple's R&D spending?", "type": "single_doc"},
    {"query": "What litigation risks does Meta disclose?", "type": "single_doc"},
    {"query": "How did Microsoft's cybersecurity risk disclosures change over time?", "type": "cross_year"},
    {"query": "How has Google's AI-related risk language evolved across recent filings?", "type": "cross_year"},
    {"query": "Compare how these companies describe risks from AI competition.", "type": "cross_company"},
    {"query": "Compare regulatory and antitrust risk across these companies.", "type": "cross_company"},
    {"query": "What do these companies say about reliance on cloud infrastructure?", "type": "cross_company"},
]


# --- Run all queries against both indexes ---
rows = []
for tq in test_queries:
    query_vector = embed_query(tq["query"])

    fixed_results = run_query(fixed_index, query_vector)
    section_results = run_query(section_index, query_vector)

    for rank, r in enumerate(fixed_results, start=1):
        rows.append({
            "query": tq["query"], "query_type": tq["type"], "strategy": "fixed_size",
            "rank": rank, "chunk_id": r[0], "company": r[1], "fiscal_year": r[2],
            "chunk_text": r[3], "score": r[4]
        })

    for rank, r in enumerate(section_results, start=1):
        rows.append({
            "query": tq["query"], "query_type": tq["type"], "strategy": "section_aware",
            "rank": rank, "chunk_id": r[0], "company": r[1], "fiscal_year": r[2],
            "chunk_text": r[3], "score": r[4]
        })

results_df = pd.DataFrame(rows)
spark_results_df = spark.createDataFrame(results_df)
spark_results_df.write.format("delta").mode("overwrite").saveAsTable("rag_pipeline.main.eval_retrieval_results")

print("Total rows logged:", len(rows))

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

In [0]:
summary_df = (
    spark.table("rag_pipeline.main.eval_retrieval_results")
    .groupBy("strategy")
    .agg(
        {"score": "avg"}
    )
    .withColumnRenamed("avg(score)", "avg_score_across_all_ranks")
)
display(summary_df)

strategy,avg_score_across_all_ranks
fixed_size,0.6298755937499998
section_aware,0.6299461460000001


In [0]:
# Average of just the top-1 result per query — often more meaningful than averaging all 5
top1_df = (
    spark.table("rag_pipeline.main.eval_retrieval_results")
    .filter("rank = 1")
    .groupBy("strategy")
    .agg({"score": "avg"})
    .withColumnRenamed("avg(score)", "avg_top1_score")
)
display(top1_df)

strategy,avg_top1_score
fixed_size,0.64291581125
section_aware,0.639891425


In [0]:
# Variances across query types

by_type_df = (
    spark.table("rag_pipeline.main.eval_retrieval_results")
    .filter("rank = 1")
    .groupBy("strategy", "query_type")
    .agg({"score": "avg"})
    .withColumnRenamed("avg(score)", "avg_top1_score")
    .orderBy("query_type", "strategy")
)
display(by_type_df)

strategy,query_type,avg_top1_score
fixed_size,cross_company,0.6358104366666667
section_aware,cross_company,0.6323191
fixed_size,cross_year,0.6530492999999999
section_aware,cross_year,0.6460869
fixed_size,single_doc,0.6432655266666667
section_aware,single_doc,0.6433334333333334


In [0]:
import mlflow

mlflow.set_experiment("/Shared/rag_pipeline_chunking_comparison")

for strategy in ["fixed_size", "section_aware"]:
    with mlflow.start_run(run_name=f"chunking_{strategy}"):
        strategy_df = spark.table("rag_pipeline.main.eval_retrieval_results").filter(f"strategy = '{strategy}'")

        avg_score_all = strategy_df.agg({"score": "avg"}).collect()[0][0]
        avg_top1 = strategy_df.filter("rank = 1").agg({"score": "avg"}).collect()[0][0]

        if strategy == "fixed_size":
            total_chunks, avg_len, noise_count = 6457, 998.5, 2
        else:
            total_chunks, avg_len, noise_count = 7197, 870.3, 124

        mlflow.log_param("chunking_strategy", strategy)
        mlflow.log_param("chunk_size", 1000)
        mlflow.log_param("chunk_overlap", 150)
        mlflow.log_metric("total_chunks", total_chunks)
        mlflow.log_metric("avg_chunk_length", avg_len)
        mlflow.log_metric("noise_fragments_dropped", noise_count)
        mlflow.log_metric("avg_retrieval_score_all_ranks", avg_score_all)
        mlflow.log_metric("avg_top1_retrieval_score", avg_top1)

        results_pd = strategy_df.toPandas()
        results_pd.to_csv(f"/tmp/{strategy}_eval_results.csv", index=False)
        mlflow.log_artifact(f"/tmp/{strategy}_eval_results.csv")

print("Logged both strategies to MLflow. Check the Experiments tab in the left sidebar.")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


Logged both strategies to MLflow. Check the Experiments tab in the left sidebar.
